# agentic-verifier-selfrefine — full run, from scratch

Run every cell in order, top to bottom. Nothing from an earlier session is needed or
used: the zip carries the code and datasets, and every result below is produced here.

Each cell either succeeds or stops the run. Four of them exist only to stop it early:

- **7, smoke test.** Trains, merges, saves, loads and generates once at the smallest
  size, about four minutes, so a library incompatibility shows up now and not two
  hours in.
- **8, split gate.** Refuses to continue if the training data overlaps the benchmark.
- **9, clean slate.** Refuses to continue if outputs or models from another run are
  already on disk.
- **16, selection gate.** Refuses to continue if no working checkpoint was found.

| section | what | hardware | time |
|---|---|---|---|
| 1–6 | upload, pinned install, tests | CPU | ~6 min |
| 7 | smoke test | GPU | ~4 min |
| 8–10 | gates, datasets, synthetic experiment | CPU | ~4 min |
| 11–12 | four training passes + summary | GPU | ~60 min |
| 13–14 | agent loop + five-condition ladder, on the last pass's model | GPU | ~75 min |
| 15–16 | select a checkpoint that did not collapse, score it | GPU | ~15–55 min |
| 17–18 | agent loop + ladder again, on the selected model | GPU | ~75 min |
| 19 | final download | – | – |

About four to five hours on a T4. A backup zip is downloaded after every GPU stage, so
a dropped session costs at most the stage that was running. See the end for resuming.

Runtime → Change runtime type → T4 GPU before starting.


## 1. Check the GPU


In [1]:
!nvidia-smi


Sat Nov 14 22:32:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload the project zip

Earlier uploads are removed first. Colab saves a second upload of the same filename as
`name (1).zip`, and unpacking would then quietly reopen the old one.


In [2]:
import glob, os
from google.colab import files

for z in glob.glob('/content/agentic-verifier-selfrefine*.zip'):
    os.remove(z)

uploaded = files.upload()          # pick agentic-verifier-selfrefine.zip
print('uploaded:', list(uploaded))


Saving agentic-verifier-selfrefine.zip to agentic-verifier-selfrefine.zip
uploaded: ['agentic-verifier-selfrefine.zip']


## 3. Unpack


In [3]:
import glob, os, shutil

zips = sorted(glob.glob('/content/agentic-verifier-selfrefine*.zip'), key=os.path.getmtime)
assert zips, 'no zip found -- run cell 2 first'
ZIP_PATH = zips[-1]
print('unpacking', ZIP_PATH)

shutil.rmtree('/content/agentic-verifier-selfrefine', ignore_errors=True)
!unzip -q -o "{ZIP_PATH}" -d /content
os.chdir('/content/agentic-verifier-selfrefine')
for needed in ('src/smoke_test.py', 'src/select_checkpoint.py', 'requirements-colab.txt'):
    assert os.path.exists(needed), f'{needed} missing -- this is an older zip'
!ls


unpacking /content/agentic-verifier-selfrefine.zip
configs     PROVENANCE.md  requirements-colab.txt  scripts
data	    pytest.ini	   requirements-dev.txt    slurm
Dockerfile  README.md	   requirements.txt	   src
notebooks   report	   results		   tests


## 4. Install the pinned GPU stack

`requirements-colab.txt` pins `trl`, `peft` and `transformers` to the versions the
recorded runs used. An unpinned upgrade installs whatever is newest today, and the
training scripts sit directly on APIs that change between releases.

`torch` is left alone on purpose: Colab ships a build matched to its own CUDA runtime,
and replacing it is the install most likely to break the GPU.

`torchao` is removed, not upgraded. Colab preinstalls a version `peft` rejects, nothing
here uses it, and its newer wheels are compiled against one specific `torch`.


In [4]:
!pip -q uninstall -y torchao
!pip -q install -r /content/agentic-verifier-selfrefine/requirements-colab.txt -r /content/agentic-verifier-selfrefine/requirements-dev.txt
assert globals().get('_exit_code', 0) == 0, 'pip install failed -- stop and send the output above'
!pip freeze > /content/agentic-verifier-selfrefine/results/raw/pip_freeze.txt
!python -c "import torch, transformers, trl, peft, accelerate, datasets; print('torch', torch.__version__, '| transformers', transformers.__version__, '| trl', trl.__version__, '| peft', peft.__version__, '| accelerate', accelerate.__version__, '| datasets', datasets.__version__)"
assert globals().get('_exit_code', 0) == 0, 'the GPU stack does not import -- stop and send the output above'


torch 2.14.1+cu130 | transformers 5.17.0 | trl 1.13.0 | peft 0.21.0 | accelerate 1.15.0 | datasets 4.8.5


## 5. Helpers: stop on failure, back up after every stage

`check()` stops the run when the shell command above it failed. Without it, a failed
stage is just red text and the next cell starts anyway.

`backup()` zips `results/` without the model weights and hands it to the browser. The
first time, the browser asks whether this page may download multiple files: allow it.
Set `USE_DRIVE = True` to also copy every backup to Google Drive, which survives a
closed tab; it asks for permission once, here.


In [5]:
import os, shutil, subprocess
from google.colab import files

ROOT = '/content/agentic-verifier-selfrefine'
USE_DRIVE = False

DRIVE_DIR = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/agentic-verifier-selfrefine-backups'
    os.makedirs(DRIVE_DIR, exist_ok=True)

def check(stage):
    code = globals().get('_exit_code', 0)
    assert code == 0, f'{stage} FAILED (exit {code}). Stop here and send the output above.'
    print(f'OK: {stage}')

def backup(tag):
    os.makedirs('/content/out', exist_ok=True)
    path = f'/content/out/results_{tag}.zip'
    if os.path.exists(path):
        os.remove(path)
    subprocess.run(['zip', '-rq', path, 'results', '-x', 'results/sft-merged-model/*',
                    'results/lora-*/*', 'results/selected-checkpoint/*'], cwd=ROOT, check=True)
    print(f'backup: {path} ({os.path.getsize(path) / 1e6:.1f} MB)')
    if DRIVE_DIR:
        shutil.copy(path, DRIVE_DIR)
        print('  copied to Drive')
    files.download(path)

print('helpers ready | Drive backups:', 'on' if DRIVE_DIR else 'off')


helpers ready | Drive backups: off


**Only when resuming in a new session:** set `RESTORE = True` and pick the newest
`results_*.zip` you downloaded. It is unpacked over `results/`, so the final zip still
holds everything. Leave it `False` on a first run.


In [6]:
RESTORE = False
if RESTORE:
    up = files.upload()
    name = os.path.abspath(list(up)[0])
    subprocess.run(['unzip', '-q', '-o', name, '-d', ROOT], check=True)
    print('restored', name)
    print(sorted(os.listdir(os.path.join(ROOT, 'results', 'raw'))))


## 6. Run the test suite

Must be green before anything is trained. It checks the verifier's verdicts and
accepted answer formats, cross-process determinism, the experiment's own validity
conditions, the checkpoint-selection rule, and that the datasets do not overlap the
benchmark.


In [7]:
!cd /content/agentic-verifier-selfrefine && python -m pytest
check('test suite')


........................................................................ [ 42%]
.................ss..................................................... [ 84%]
...........................                                              [100%]
169 passed, 2 skipped in 65.83s (0:01:05)
OK: test suite


## 7. Smoke test: the whole chain once, as small as it goes

Runs the real training scripts for one epoch each, merges, saves, loads both models
back, generates, and grades. About four minutes. Its answers come from an
under-trained model and say nothing about quality; the only question is whether every
library call in the chain still works. It removes the checkpoints it made.

**If this fails, do not continue.** Send the output.


In [8]:
!cd /content/agentic-verifier-selfrefine/src && python smoke_test.py
check('smoke test')


Installed versions
  accelerate     1.15.0
  datasets       4.8.5
  peft           0.21.0   (pinned 0.21.0)
  sympy          1.14.0
  torch          2.14.1+cu130
  transformers   5.17.0   (pinned 5.17.0)
  trl            1.13.0   (pinned 1.13.0)

GPU: Tesla T4 | CUDA 13.0 | torch 2.14.1+cu130

--- SFT train + merge + save: train_lora_sft.py (1 epoch)
GPU: Tesla T4
config.json: 100% 659/659 [00:00<00:00, 2.24MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 19.1MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 59.8MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 108MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 154MB/s]

model.safetensors: downloading bytes:   3% 31.0M/988M [00:00<00:21, 43.9MB/s]
model.safetensors: downloading bytes:  17% 173M/988M [00:01<00:03, 229MB/s, 13.2MB/s  ]
model.safetensors: downloading bytes:  22% 215M/988M [00:01<00:03, 220MB/s, 19.0MB/s  ]
model.safetensors: downloading bytes:  31% 305M/988M [00:01<00:03, 216MB/s, 27.1MB/s  ]
model.safetenso

## 8. Gate: the split must be disjoint

A different seed is not a disjoint sample. This measures the overlap by equation,
which is the only comparison that means anything, and stops if it is not zero.

Do not compare by `problem_id`. Ids are positional within a draw, so the benchmark and
the training draw share `linear_0000`..`linear_0014` while none of them is the same
equation.


In [9]:
import sys
sys.path.insert(0, '/content/agentic-verifier-selfrefine/src')
import splits
from tasks import contamination

held = splits.held_out()
for n, name in [(100, 'SFT draw'), (60, 'DPO draw')]:
    report = contamination(splits.training(n), held)
    print(f"{name:10} {report['n_overlapping']}/{report['n_held_out']} overlapping equations")
    assert report['n_overlapping'] == 0, 'split is contaminated; stop here'
assert contamination(splits.validation(), held)['n_overlapping'] == 0

print(f'\nbenchmark: {len(held)} problems, seed {splits.HELD_OUT_SEED}')
print('GATE PASSED — training and validation are disjoint from the benchmark')


SFT draw   0/45 overlapping equations
DPO draw   0/45 overlapping equations

benchmark: 45 problems, seed 99
GATE PASSED — training and validation are disjoint from the benchmark


## 9. Gate: clean slate

Every result of this run must come from this run. The zip ships no model weights and
no outputs at the paths the stages write to; this confirms it, so that nothing below
can be an old file read as a new one. On a resumed session (`RESTORE = True` above) the
restored outputs are listed and allowed.


In [10]:
import glob, os

models = [p for p in ('results/sft-merged-model', 'results/lora-sft-adapter',
                      'results/lora-dpo-zeroshot-adapter', 'results/selected-checkpoint')
          if os.path.exists(os.path.join(ROOT, p))]
assert not models, (f'model weights already on disk: {models}. At the START of a run: unpack a '
                    'fresh zip (cell 3). In the MIDDLE of a run: skip this cell, it has already passed.')

outputs = sorted(os.path.relpath(p, ROOT) for pattern in (
    'results/raw/eval/*', 'results/llm_agent_experiment*',
    'results/feedback_protocol_ablation*', 'results/raw/checkpoint_selection.json')
    for p in glob.glob(os.path.join(ROOT, pattern)))
if globals().get('RESTORE', False):
    print(f'resumed session: {len(outputs)} restored output files kept')
else:
    assert not outputs, f'outputs from another run are present: {outputs[:5]}'
print('GATE PASSED — nothing stale on disk')


GATE PASSED — nothing stale on disk


## 10. Datasets and the synthetic experiment (CPU)

Rebuilds both training files and runs the CPU turn-budget experiment. The builders
print the contamination as they write. Confirm `0/45` in the output.


In [11]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py data cpu
check('datasets + CPU experiment')


Stages: ['data', 'cpu']
Rough budget: ~3 min
GPU available: True


[data] build_sft_data.py
split: 300 training / 45 held-out, 0 overlapping equations (the unfiltered draw would have overlapped 6: {'linear': 5, 'rational': 1})
300 SFT examples written to data/sft_pairs.jsonl
held-out contamination: 0/45
{
  "problem_id": "linear_0000",
  "family": "linear",
  "equation_str": "Eq(x + 3, 0)",
  "messages": [
    {
      "role": "system",
      "content": "You are a careful algebra solver. Show brief work, then give the final answer line."
    },
    {
      "role": "user",
      "content": "Solve for x: Eq(x + 3, 0)\nGive the final answer on its own line as 'x = v1, v2' or 'x = no solution'."
    },
    {
      "role": "assistant",
      "content": "This is linear. Moving the constant term gives 1 times the unknown equals -3. Dividing both sides by 1 isolates the unknown.\nx = -3"
    }
  ]
}
[data] build_sft_data.py finished in 9.6s

[data] build_dpo_pairs_zeroshot.py
split: 180 trainin

## 11. Four training passes

`sft dpo eval`, never `eval` alone. Decoding is greedy and therefore deterministic, so
repeating the evaluation on one checkpoint prints the same table four times and
measures nothing. Each pass here trains a fresh model, and the spread between passes
is the GPU non-determinism, which is the quantity in question: one pass is a draw, not
a measurement.

About 60 minutes.


In [12]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py sft dpo eval --repeat=4
backup('1_after_passes')
check('four training passes')


Stages: ['sft', 'dpo', 'eval'] x4
Rough budget: ~52 min
GPU available: True


########## pass 1 of 4 ##########

[sft] train_lora_sft.py
GPU: Tesla T4
Loading weights: 100% 290/290 [00:00<00:00, 16919.34it/s]
Tokenizing train dataset: 100% 255/255 [00:00<00:00, 1384.02 examples/s]
Building labels for train dataset: 100% 255/255 [00:00<00:00, 4131.52 examples/s]
Truncating train dataset: 100% 255/255 [00:00<00:00, 5193.64 examples/s]
Dropping fully masked examples from train dataset: 100% 255/255 [00:00<00:00, 7399.15 examples/s]
Tokenizing eval dataset: 100% 45/45 [00:00<00:00, 1222.37 examples/s]
Building labels for eval dataset: 100% 45/45 [00:00<00:00, 3295.45 examples/s]
Truncating eval dataset: 100% 45/45 [00:00<00:00, 4139.84 examples/s]
Dropping fully masked examples from eval dataset: 100% 45/45 [00:00<00:00, 5606.19 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

OK: four training passes


## 12. Combine the passes

Never read one pass on its own. This prints every pass, the spread, the DPO delta per
pass, the verifier's reason for each failure, and which problems move between passes.


In [13]:
!cd /content/agentic-verifier-selfrefine/src && python summarize_eval.py
check('summary of the passes')


HELD-OUT EVALUATION -- 4 passes, 45 problems, greedy decoding
                            pass 1    pass 2    pass 3    pass 4    summary
--------------------------------------------------------------------------
                    Base     0.000     0.000     0.000     0.000    mean 0.000   sd 0.000   range 0.000-0.000
              + LoRA-SFT     0.311     0.289     0.311     0.333    mean 0.311   sd 0.018   range 0.289-0.333
   + LoRA-SFT + LoRA-DPO     0.289     0.267     0.400     0.244    mean 0.300   sd 0.069   range 0.244-0.400
               DPO delta    -0.022    -0.022    +0.089    -0.089    mean -0.011   sd 0.074   range -0.089-0.089

  DPO helped in 1 of 4 passes, hurt in 3.
  The sign is not stable across passes, so the stage has no established direction on this setup.

VERIFIER REASON, mean count per pass
            reason |       base |        SFT |    SFT+DPO
-----------------------------------------------------------
           correct |        0.0 |       14.0 |   

## 13. The agent loop, on the last pass's model

Verifier feedback against a re-sampling control, paired under identical sampling
seeds. About 25 minutes.

This runs on whatever model pass 4 left behind, which may be a collapsed one. That is
intended. Sections 17–18 repeat both experiments on a model known to work, and the
pair shows whether the benefit of feedback depends on the model being damaged.


In [14]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py agent
backup('2_after_first_agent')
check('agent loop on the last-pass model')


Stages: ['agent']
Rough budget: ~50 min
GPU available: True


[agent] run_llm_agent_experiment.py
Loading weights: 100% 290/290 [00:00<00:00, 370.70it/s]
Running on: cuda | model: sft_dpo | 45 problems x 3 seeds x 2 conditions
Model under test (re-training the pipeline replaces this):
  sft_merged: ../results/sft-merged-model  (last modified 2026-11-14T23:21:08)
  dpo_adapter: ../results/lora-dpo-zeroshot-adapter  (last modified 2026-11-14T23:24:37)
  [seed 0/feedback] 15/45 problems
  [seed 0/feedback] 30/45 problems
  [seed 0/feedback] 45/45 problems
  seed 0 feedback: turn<=1: 0.289, turn<=2: 0.333, turn<=3: 0.400
  [seed 0/control] 15/45 problems
  [seed 0/control] 30/45 problems
  [seed 0/control] 45/45 problems
  seed 0 control: turn<=1: 0.289, turn<=2: 0.311, turn<=3: 0.333
  [seed 1/feedback] 15/45 problems
  [seed 1/feedback] 30/45 problems
  [seed 1/feedback] 45/45 problems
  seed 1 feedback: turn<=1: 0.267, turn<=2: 0.289, turn<=3: 0.422
  [seed 1/control] 15/45 problems
  [

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

OK: agent loop on the last-pass model


## 14. The five-condition feedback ladder, on the same model

`retry` / `generic` / `diagnostic` / `actionable` / `constrained`. Separates “feedback
does not help” from “feedback was delivered badly”. About 50 minutes.


In [15]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py ablation
backup('3_after_first_ladder')
check('feedback ladder on the last-pass model')


Stages: ['ablation']
Rough budget: ~45 min
GPU available: True


[ablation] feedback_protocol_ablation.py
Device: cuda | model: sft_dpo
Conditions: ['retry', 'generic', 'diagnostic', 'actionable', 'constrained']
Budget: 5 x 45 problems x 3 turns x 3 seeds = 2025 generations
  ~44 min on a T4 at ~1.3 s/generation

Loading weights: 100% 290/290 [00:00<00:00, 402.95it/s]
  [retry seed 0] 15/45
  [retry seed 0] 30/45
  [retry seed 0] 45/45
  [retry seed 1] 15/45
  [retry seed 1] 30/45
  [retry seed 1] 45/45
  [retry seed 2] 15/45
  [retry seed 2] 30/45
  [retry seed 2] 45/45
  [generic seed 0] 15/45
  [generic seed 0] 30/45
  [generic seed 0] 45/45
  [generic seed 1] 15/45
  [generic seed 1] 30/45
  [generic seed 1] 45/45
  [generic seed 2] 15/45
  [generic seed 2] 30/45
  [generic seed 2] 45/45
  [diagnostic seed 0] 15/45
  [diagnostic seed 0] 30/45
  [diagnostic seed 0] 45/45
  [diagnostic seed 1] 15/45
  [diagnostic seed 1] 30/45
  [diagnostic seed 1] 45/45
  [diagnostic seed 2] 15/45
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

OK: feedback ladder on the last-pass model


## 15. Select a checkpoint that did not collapse

The DPO stage can wreck the model it is given. This retrains until it has not: the
SFT+DPO rate must be within three problems of the same attempt's SFT-only rate, both
measured on a **validation** set of 45 problems that shares nothing with the benchmark.
The benchmark is never consulted, because retraining until the test score looks good
is selection on the test set.

Every attempt is logged, not only the survivor. The number of attempts needed is a
direct estimate of the collapse rate, which is a finding in its own right.

About 8 minutes per attempt, at most six attempts.


In [16]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py select
backup('4_after_selection')


Stages: ['select']
Rough budget: ~25 min
GPU available: True


[select] select_checkpoint.py
Selecting on 45 VALIDATION problems (seed 7); the 45-problem benchmark is not consulted.
Keeping the first checkpoint whose SFT+DPO validation rate is within 0.075 of its own SFT-only rate.
At most 6 attempts.

attempt 1 of 6

--- train_lora_sft.py
GPU: Tesla T4
Loading weights: 100% 290/290 [00:00<00:00, 4258.15it/s]
Tokenizing train dataset: 100% 255/255 [00:00<00:00, 1139.04 examples/s]
Building labels for train dataset: 100% 255/255 [00:00<00:00, 4040.71 examples/s]
Truncating train dataset: 100% 255/255 [00:00<00:00, 4521.37 examples/s]
Dropping fully masked examples from train dataset: 100% 255/255 [00:00<00:00, 6929.63 examples/s]
Tokenizing eval dataset: 100% 45/45 [00:00<00:00, 814.10 examples/s]
Building labels for eval dataset: 100% 45/45 [00:00<00:00, 1635.22 examples/s]
Truncating eval dataset: 100% 45/45 [00:00<00:00, 2474.81 examples/s]
Dropping fully masked examples from eval da

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 16. Gate, then score the selected checkpoint once

Stops the run if selection kept nothing, because the model on disk would then be a
collapsed one and everything below would be mislabelled. It also moves the outputs of
sections 13–14 into `results/raw/before_selection/`: the agent loop and the ladder
write to fixed filenames, and the re-run would otherwise overwrite them.


In [ ]:
import glob, json, os, shutil

log_path = os.path.join(ROOT, 'results/raw/checkpoint_selection.json')
assert os.path.exists(log_path), 'section 15 did not finish; run it first'
log = json.load(open(log_path))
for a in log['attempts']:
    print(f"attempt {a['attempt']}: SFT {a['sft_validation']:.3f} | SFT+DPO "
          f"{a['sft_dpo_validation']:.3f} | {'collapsed' if a['collapsed'] else 'kept'}")
assert log['kept'] is not None, (
    'no checkpoint survived selection -- the model on disk is a collapsed one; stop here')
print(f"\n{log['n_collapsed']} of {log['n_attempts']} attempts collapsed")

keep = os.path.join(ROOT, 'results/raw/before_selection')
os.makedirs(keep, exist_ok=True)
moved = []
for pattern in ('results/llm_agent_experiment*', 'results/feedback_protocol_ablation*'):
    for path in glob.glob(os.path.join(ROOT, pattern)):
        shutil.move(path, os.path.join(keep, os.path.basename(path)))
        moved.append(os.path.basename(path))
print(f'moved {len(moved)} earlier output files to results/raw/before_selection/')
print('GATE PASSED — a working checkpoint is in place')


The selected checkpoint's score goes to `results/raw/selected_eval/`, apart from the
four passes in `results/raw/eval/`. A chosen model is not a fifth draw from the same
distribution, and the two must never be averaged together.


In [18]:
!cd /content/agentic-verifier-selfrefine/src && AVS_EVAL_DIR=../results/raw/selected_eval python rerun_all.py eval && python summarize_eval.py --dir ../results/raw/selected_eval
backup('5_after_selected_eval')
check('benchmark score of the selected checkpoint')


Stages: ['eval']
Rough budget: ~6 min
GPU available: True


[eval] evaluate_redesign_v2.py
Running on: cuda
Evaluating BASE model zero-shot...
Loading weights: 100% 290/290 [00:00<00:00, 296.25it/s]
  [base] 5/45 done, 0 correct so far
  [base] 10/45 done, 0 correct so far
  [base] 15/45 done, 0 correct so far
  [base] 20/45 done, 0 correct so far
  [base] 25/45 done, 0 correct so far
  [base] 30/45 done, 0 correct so far
  [base] 35/45 done, 0 correct so far
  [base] 40/45 done, 0 correct so far
  [base] 45/45 done, 0 correct so far
Base model zero-shot solve rate: 0.000
Evaluating SFT-only model zero-shot...
Loading weights: 100% 290/290 [00:00<00:00, 316.33it/s]
  [sft] 5/45 done, 5 correct so far
  [sft] 10/45 done, 9 correct so far
  [sft] 15/45 done, 13 correct so far
  [sft] 20/45 done, 13 correct so far
  [sft] 25/45 done, 13 correct so far
  [sft] 30/45 done, 14 correct so far
  [sft] 35/45 done, 15 correct so far
  [sft] 40/45 done, 15 correct so far
  [sft] 45/45 done, 15 co

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

OK: benchmark score of the selected checkpoint


## 17. The agent loop, on the selected checkpoint

If the benefit of feedback survives on a model that works, the central claim stands
without its largest caveat. If it shrinks or disappears, that is equally worth knowing.


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py agent
backup('6_after_selected_agent')
check('agent loop on the selected checkpoint')


In [20]:
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/agentic_backups'
os.makedirs(DRIVE_DIR, exist_ok=True)

!cp -v /content/out/*.zip "$DRIVE_DIR"/
!cp -rv /content/agentic-verifier-selfrefine/results "$DRIVE_DIR"/results_snapshot

Mounted at /content/drive
'/content/out/results_1_after_passes.zip' -> '/content/drive/MyDrive/agentic_backups/results_1_after_passes.zip'
'/content/out/results_2_after_first_agent.zip' -> '/content/drive/MyDrive/agentic_backups/results_2_after_first_agent.zip'
'/content/out/results_3_after_first_ladder.zip' -> '/content/drive/MyDrive/agentic_backups/results_3_after_first_ladder.zip'
'/content/out/results_4_after_selection.zip' -> '/content/drive/MyDrive/agentic_backups/results_4_after_selection.zip'
'/content/out/results_5_after_selected_eval.zip' -> '/content/drive/MyDrive/agentic_backups/results_5_after_selected_eval.zip'
'/content/out/results_6_after_selected_agent.zip' -> '/content/drive/MyDrive/agentic_backups/results_6_after_selected_agent.zip'
'/content/agentic-verifier-selfrefine/results' -> '/content/drive/MyDrive/agentic_backups/results_snapshot'
'/content/agentic-verifier-selfrefine/results/solve_rate_by_turn_budget.png' -> '/content/drive/MyDrive/agentic_backups/results_sn

In [21]:
# This cell tests if your browser allows multiple automatic downloads.
# Running this should download two separate text files to your computer.
from google.colab import files
import os

# Create two small test files
with open('download_test_1.txt', 'w') as f:
    f.write('Browser Download Test File 1 - Success!')

with open('download_test_2.txt', 'w') as f:
    f.write('Browser Download Test File 2 - Success!')

print("Triggering downloads... If your browser asks for permission, choose 'Allow'.")

# Trigger downloads back-to-back
files.download('download_test_1.txt')
files.download('download_test_2.txt')


Triggering downloads... If your browser asks for permission, choose 'Allow'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 18. The feedback ladder, on the selected checkpoint


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py ablation
backup('7_after_selected_ladder')
check('feedback ladder on the selected checkpoint')


Stages: ['ablation']
Rough budget: ~45 min
GPU available: True


[ablation] feedback_protocol_ablation.py
Device: cuda | model: sft_dpo
Conditions: ['retry', 'generic', 'diagnostic', 'actionable', 'constrained']
Budget: 5 x 45 problems x 3 turns x 3 seeds = 2025 generations
  ~44 min on a T4 at ~1.3 s/generation

Loading weights: 100% 290/290 [00:01<00:00, 213.13it/s]
  [retry seed 0] 15/45
  [retry seed 0] 30/45
  [retry seed 0] 45/45
  [retry seed 1] 15/45
  [retry seed 1] 30/45
  [retry seed 1] 45/45
  [retry seed 2] 15/45
  [retry seed 2] 30/45
  [retry seed 2] 45/45
  [generic seed 0] 15/45
  [generic seed 0] 30/45
  [generic seed 0] 45/45
  [generic seed 1] 15/45
  [generic seed 1] 30/45
  [generic seed 1] 45/45
  [generic seed 2] 15/45
  [generic seed 2] 30/45
  [generic seed 2] 45/45
  [diagnostic seed 0] 15/45
  [diagnostic seed 0] 30/45
  [diagnostic seed 0] 45/45
  [diagnostic seed 1] 15/45
  [diagnostic seed 1] 30/45
  [diagnostic seed 1] 45/45
  [diagnostic seed 2] 15/45
 

## 19. Final download

One zip with everything this session produced: the four passes, the first agent and
ladder runs (`raw/before_selection/`), the selection log, the selected checkpoint's
benchmark score, both experiments re-run on it, a run manifest per stage, the smoke
test record and the installed package list.


In [ ]:
backup('final')


In [22]:
import os, shutil

def backup(name):
    os.makedirs('/content/out', exist_ok=True)
    zip_path = shutil.make_archive(
        f'/content/out/results_{name}', 'zip',
        root_dir='/content/agentic-verifier-selfrefine',
        base_dir='results'
    )
    shutil.copy(zip_path, DRIVE_DIR)
    size = os.path.getsize(zip_path) / 1e6
    print(f'backup: {zip_path} ({size:.1f} MB) -> saved to Drive')

backup('test')

backup: /content/out/results_test.zip (1839.4 MB) -> saved to Drive


---

### What to send back

`results_final.zip`, and this notebook with its outputs (File → Download → Download
.ipynb). The numbers are **not** promoted into `results/recorded/manifest.json`
automatically. Deciding that a number is now the project's result is a judgement made
after looking at it, not a side effect of the script that produced it.

### If the session is cut off

The Colab disk is lost with the session, the downloaded backups are not. In a new
session run cells 1–5, set `RESTORE = True` and pick the newest backup, run 6–9, then:

| cut off during | continue with |
|---|---|
| 11 (passes) | 11 again; passes accumulate and 12 summarises all of them |
| 13 or 14 | 15. The last pass's model is gone, so 13–14 are skipped; 17–18 still run |
| 15 to 18 | 15 again. Selection retrains, so it needs nothing from before |

### If a cell fails

| symptom | cause |
|---|---|
| `SMOKE TEST FAILED at: ...` | a library call no longer works on this stack; send the output |
| `pip install failed` | a pinned version could not be installed; send the output |
| `stage 'X' needs [...]` | an earlier stage did not run; the driver refuses to evaluate a missing or stale model |
| `--repeat=N was given, but none of [...] retrains` | use `sft dpo eval --repeat=N` |
| `split is contaminated` | the split gate failed; do not train |
| `outputs from another run are present` | the zip was not unpacked fresh; re-run cell 3 |
| `no checkpoint survived selection` | every attempt collapsed; stop and send the log |
| `older zip` | cell 2 received a previous version of the project zip |
| `torch.cuda.is_available() is False` | Runtime → Change runtime type → T4 GPU |


In [23]:
!du -sh /content/agentic-verifier-selfrefine/* /content/out/* 2>/dev/null

16K	/content/agentic-verifier-selfrefine/configs
416K	/content/agentic-verifier-selfrefine/data
4.0K	/content/agentic-verifier-selfrefine/Dockerfile
4.0K	/content/agentic-verifier-selfrefine/download_test_1.txt
4.0K	/content/agentic-verifier-selfrefine/download_test_2.txt
40K	/content/agentic-verifier-selfrefine/notebooks
32K	/content/agentic-verifier-selfrefine/PROVENANCE.md
4.0K	/content/agentic-verifier-selfrefine/pytest.ini
40K	/content/agentic-verifier-selfrefine/README.md
296K	/content/agentic-verifier-selfrefine/report
4.0K	/content/agentic-verifier-selfrefine/requirements-colab.txt
4.0K	/content/agentic-verifier-selfrefine/requirements-dev.txt
4.0K	/content/agentic-verifier-selfrefine/requirements.txt
2.3G	/content/agentic-verifier-selfrefine/results
8.0K	/content/agentic-verifier-selfrefine/scripts
8.0K	/content/agentic-verifier-selfrefine/slurm
372K	/content/agentic-verifier-selfrefine/src
356K	/content/agentic-verifier-selfrefine/tests
620K	/content/out/results_1_after_passe

In [24]:
!du -sh /content/agentic-verifier-selfrefine/results/*

4.0K	/content/agentic-verifier-selfrefine/results/analysis.txt
48K	/content/agentic-verifier-selfrefine/results/colab_run
28K	/content/agentic-verifier-selfrefine/results/error_breakdown.png
168K	/content/agentic-verifier-selfrefine/results/fig_feedback_ladder.png
112K	/content/agentic-verifier-selfrefine/results/fig_format_degradation.png
132K	/content/agentic-verifier-selfrefine/results/fig_passk_illusion.png
88K	/content/agentic-verifier-selfrefine/results/fig_training_variance.png
4.0K	/content/agentic-verifier-selfrefine/results/llm_agent_experiment_minimal_model.json
127M	/content/agentic-verifier-selfrefine/results/lora-dpo-zeroshot-adapter
127M	/content/agentic-verifier-selfrefine/results/lora-sft-adapter
420K	/content/agentic-verifier-selfrefine/results/raw
348K	/content/agentic-verifier-selfrefine/results/recorded
1.1G	/content/agentic-verifier-selfrefine/results/selected-checkpoint
954M	/content/agentic-verifier-selfrefine/results/sft-merged-model
56K	/content/agentic-verifi